# Study 930 — When-Issued Window

**When a company splits in two, is the freshly cut-loose half a bargain?**

Between the day a break-up is announced and the day the new company actually trades
normally, there is a strange in-between zone. The child already has a price — a
*when-issued* price, good only if the split completes — but nobody quite owns it yet.
Folklore says this zone is full of free money: the parent re-rates upward on the news,
and the child gets dumped on day one by index funds and pension managers who never chose
to own it, so you can pick it up cheap.

We took **26 liquid US spin-offs (2012–2025)** and
measured all three pieces of that story against the S&P 500.

*Real-tape numbers below are frozen from [`docs/results.md`](../docs/results.md)
(Fingerprint `e78725a81ebe`, as-of 2026-06-30). Live cells run the **synthetic** control only
and say so.*

## 1. Three windows, three stories

- **The parent**, from the day after the break-up is announced to the last close before the child trades normally. Does dismantling a conglomerate re-rate what is left behind?
- **The child**, over its first 5, 10 and 21 normal trading sessions. This is the famous one: Joel Greenblatt's *forced sellers*.
- **Both halves together**, held a month. Is the sum of the parts worth more than the whole was?

Everything is measured against SPY over exactly the same calendar days, and we always wait one full session before acting — so no result here depends on being quicker than a newspaper. The figures below are *before* trading costs on purpose: the answer turns out to be a **loss**, and charging costs to a loss would only flatter the finding. Costs appear in §5, on the trade you would actually place.

In [1]:
R = dict(par_mean=-1.45, par_t=-0.26, c5_mean=-4.54, c5_t=-2.73, c21_mean=-1.02, c21_t=-0.41,
         comb_mean=1.88, comb_t=1.28)
print('parent, announcement -> distribution : %+6.2f%%  (t = %+.2f)'
      % (R['par_mean'], R['par_t']))
print('child, first 5 sessions             : %+6.2f%%  (t = %+.2f)   <-- the only real one'
      % (R['c5_mean'], R['c5_t']))
print('child, first 21 sessions            : %+6.2f%%  (t = %+.2f)'
      % (R['c21_mean'], R['c21_t']))
print('parent + child, first 21 sessions   : %+6.2f%%  (t = %+.2f)'
      % (R['comb_mean'], R['comb_t']))

parent, announcement -> distribution :  -1.45%  (t = -0.26)
child, first 5 sessions             :  -4.54%  (t = -2.73)   <-- the only real one
child, first 21 sessions            :  -1.02%  (t = -0.41)
parent + child, first 21 sessions   :  +1.88%  (t = +1.28)


## 2. The surprise: the child is not cheap, it is *falling*

Buy the child at the close of its very first normal session and hold it a week, and you lose **4.5%** to the index. Only **8 of 26** spin-offs beat SPY over that week. That is not a rounding error — it is the exact opposite of what the forced-seller story predicts.

The parent's long wait from announcement to distribution turns out to be nothing much at all (-1.45%, *t* = -0.26), and putting the two halves back together does nothing either (+1.88%, *t* = +1.28). Four of the five things we measured are noise. The fifth is real and points the wrong way.

## 3. What is probably going on

Think about who *has* to trade on the day of the distribution. Every index fund that owns
the parent is handed the child. The child immediately joins the parent's index — so those
funds cannot sell it, they have to **hold** it, and the funds that track the child's *new*
index have to **buy** it. On day one there is a wall of price-insensitive buying.

Then it stops. Over the next three sessions the child gives back
**6.1%** relative to the index, and by the end of the month it has
clawed most of that back (-0.79%). That shape — a pop, a slide, a partial
recovery — is the classic footprint of mechanical demand, not of forced selling.

> 🔬 **For the quants:** the trough sits at three sessions (-5.91%,
> *t* = -3.65, only 3/26
> positive) and is gone by a quarter (-0.74%, *t* = -0.19). Only
> 5 / 10 / 21 sessions were pre-specified, so the 3-session number carries an unpriced
> multiple-comparison penalty and is a shape diagnostic, never the headline.

## 4. Is it just a few disasters dragging the average?

No. Drop any single spin-off from the sample and the *t*-statistic stays between **-3.36** and **-2.44**. Split the sample in 2020 and it is negative in both halves (-3.77% before, -4.77% after). Spin-off children are usually smaller than the S&P 500, so we re-ran it against the small-cap and mid-cap indices too: -4.41% against IWM, -4.64% against MDY. Same answer everywhere.

## 5. So can you trade it?

Only by **shorting** — and that is where it gets awkward. Shorting a company that started
trading five days ago means borrowing shares that barely exist yet: tiny float, no settled
lending supply, most of the stock sitting in index funds. We swept the borrow cost from
free to punitive:

| You pay to borrow | You keep, per trade |
|---|--:|
| nothing (fantasy) | **+4.32%** (*t* = +2.60) |
| 25%/yr | +3.82% |
| 50%/yr | +3.33% |
| 100%/yr | +2.33% — and now the confidence interval crosses zero |

And there are only about **2 of these a year**. Two trades a year, each
one a coin-flip with a 8.5-point standard deviation, in a name your broker may
simply refuse to locate. Real effect, awful business.

## 6. Live check — the measuring stick is straight (synthetic, not the real tape)

Before believing a negative number, check the ruler. We build a **fake** world of spin-offs where we *plant* a known parent run-up and a known child drift, and run the identical code on it. It must find what we planted — and find nothing when we plant nothing.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from when_issued import data, strategy as st

# SYNTHETIC world (not the real tape) — the machinery proof.
px, ev, truth = data.synthetic_daily(signal_strength=1.0, seed=930)
planted = st.synthetic_detect(px, ev)
px0, ev0, _ = data.synthetic_daily(signal_strength=0.0, seed=930)
null = st.synthetic_detect(px0, ev0)
print('planted world: we buried a %+.2f%% child drift ->'
      ' the estimator found %+.2f%% (t = %+.2f)'
      % (truth['planted_child_alpha_21d']*100,
         planted['child21_mean_pct'], planted['child21_t']))
print('null world   : we buried nothing            ->'
      ' the estimator found %+.2f%% (t = %+.2f)'
      % (null['child21_mean_pct'], null['child21_t']))

planted world: we buried a +5.25% child drift -> the estimator found +6.63% (t = +4.43)
null world   : we buried nothing            -> the estimator found +1.15% (t = +0.81)


## Verdict

- **Signal — Real, with the sign inverted, and only just.** The child's first five
  regular-way sessions under-perform SPY by **-4.54%** (*t* = -2.73,
  HAC -3.02, bootstrap CI [-7.67, -1.42]% clear of
  zero), in both eras and against three different benchmarks. That is a genuine effect —
  it is just not the bargain the folklore advertises. The parent run-up and the
  sum-of-the-parts pop are both nothing. Two things to keep it in proportion: we measured
  five things, and after paying for all five looks the odds of seeing this by luck are
  **4.5%** — real, but a whisker inside the line, not a mile. And the sample
  is 26 hand-picked liquid spins, with 4 more lost because the
  child was later acquired and delisted; treat the size of the number with more suspicion
  than its sign.
- **Tradability — Fragile.** Harvesting it means shorting a five-day-old spin-off about
  twice a year. Survives a 50%/yr borrow (+3.33%), does not survive 100%/yr
  (+2.33%, CI through zero), and starting three sessions late turns
  +4.32% into -2.24%.